In [1]:
%load_ext autoreload
%autoreload 2
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))
import warnings
warnings.filterwarnings("ignore")

import os
import pandas as pd
import sys
sys.path.insert(0, '/home/kat/Repos/SALSA/')

### Load in the cleaned HIV dataset.

In [2]:
tokens_salsa = '#%()+-0123456789<=>BCFHILNOPRSX[]cnos'
# tokens_hiv = 'Ub)gsP5u=8H[d]noW06%2GLlO+NFSTa9ZCeI-r4(1ihAVBRc3M7pt#'

df_hiv = pd.read_csv('data/model_ready/hiv/train/anchor_smiles.csv')
tokens_hiv = list(set(list(''.join(df_hiv.smiles.values))))
print(len(tokens_hiv), len(tokens_salsa))

34 37


### Load models and get latents ... 

In [ ]:
from benchmark_utils import get_latents
from imblearn.ensemble import BalancedRandomForestClassifier as BalRF
from sklearn.model_selection import StratifiedKFold
from utilities.fp_utils import get_fps_in_parallel
import numpy as np

seedy = 666

def undo_BrCl_singles(smi):
    smi = smi.replace('R','Br')
    return smi.replace('L','Cl')
def do_BrCl_singles(smi):
    smi = smi.replace('Br','R')
    return smi.replace('Cl','L')   

# # # # # # # # # # # #
load_bs = 50
# # # # # # # # # # # #

tags = ['2022041804_04', # salsa
        '2022041807_a03', # contrastive  
        '2022041809_a04', # vanilla ae
        'morgan'
       ]

for tag in tags:
    
    if tag=='morgan':
        df_hiv = pd.read_csv('data/model_ready/hiv/train/anchor_smiles.csv')
        smis_hiv = [undo_BrCl_singles(sm) for sm in df_hiv.smiles]
        fps = get_fps_in_parallel(smis_hiv,fp_type='morgan',counts=False,bits=1024,radius=2)
        X = np.stack(fps)
    else:
        X = get_latents(tag, 'train', load_bs) 
        
    y = pd.read_csv('data/model_ready/hiv/train/labels.csv')['y'].values

    ## for holding results ... 
    df_y = pd.DataFrame(y, columns=['y_test'])

    skf = StratifiedKFold(n_splits=5, random_state=seedy, shuffle=True)
    skf.get_n_splits(X, y)

    clf = BalRF(n_estimators=100, max_features="auto",sampling_strategy='auto',
                random_state=seedy,n_jobs=-1,oob_score=True)

    oobs = []
    for i,(train_idx, test_idx) in enumerate(skf.split(X,y)):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        clf.fit(X_train,y_train)
        print(f'OOB, split {i}: {clf.oob_score_}')

        y_pred = clf.predict(X_test)
        y_prob = clf.predict_proba(X_test)
        df_y.loc[test_idx,'y_prob'] = y_prob[:,1]
        df_y.loc[test_idx,'y_pred'] = y_pred
        oobs.append(clf.oob_score_)

    df_y.to_csv(f'{tag}_hiv_benchmark.csv',index=False)
    print(f"Mean OOB: {sum(oobs)/5}")

Using 4 GPUs!
Loaded model weights from /home/kat/Repos/SALSA/results/models/2022041804_04/29.pt!


 11%|█▏        | 73/636 [00:19<01:42,  5.47it/s] 

### Fingerprint baseline.

In [ ]:
# from utilities.fp_utils import get_fp
# def undo_BrCl_singles(smi):
#     smi = smi.replace('R','Br')
#     return smi.replace('L','Cl')
# def do_BrCl_singles(smi):
#     smi = smi.replace('Br','R')
#     return smi.replace('Cl','L')   

# for row in df_hiv.itertuples():
#     _smi = row.smiles
#     smi = undo_BrCl_singles(_smi)
# #     print(smi)
#     fp = get_fp(smi)
# #     print(fp.shape)
#     if fp.shape[0] != 1024:
#         print(smi, fp.shape)
#         break

In [ ]:
# from utilities.fp_utils import get_fps_in_parallel
# # import numpy as np

# df_hiv = pd.read_csv('data/model_ready/hiv/train/anchor_smiles.csv')
# smis_hiv = [undo_BrCl_singles(sm) for sm in df_hiv.smiles]

# fps = get_fps_in_parallel(smis_hiv,fp_type='morgan',counts=False,bits=1024,radius=2)
# X = np.stack(fps)
# display(fps.shape)

In [ ]:
# # # # # # #
# tag = 'morgan'
# # # # # # #

# y = pd.read_csv('data/model_ready/hiv/train/labels.csv')['y'].values
# df_y = pd.DataFrame(y, columns=['y_test'])

# skf = StratifiedKFold(n_splits=5, random_state=seedy, shuffle=True)
# skf.get_n_splits(X, y)
# clf = BalRF(n_estimators=100, max_features="auto",sampling_strategy='auto',
#             random_state=seedy,n_jobs=-1,oob_score=True)
# oobs = []
# for i,(train_idx, test_idx) in enumerate(skf.split(X,y)):
#     X_train, X_test = X[train_idx], X[test_idx]
#     y_train, y_test = y[train_idx], y[test_idx]
#     clf.fit(X_train,y_train)
#     print(f'OOB, split {i}: {clf.oob_score_}')

#     y_pred = clf.predict(X_test)
#     y_prob = clf.predict_proba(X_test)
#     df_y.loc[test_idx,'y_prob'] = y_prob[:,1]
#     df_y.loc[test_idx,'y_pred'] = y_pred
#     oobs.append(clf.oob_score_)

# df_y.to_csv(f'{tag}_hiv_benchmark.csv',index=False)
# print(f"Mean OOB: {sum(oobs)/5}")

In [ ]:
from benchmark_utils import get_metrics
import pprint

versions = ['salsa','contrastive','vanilla AE','morgan']

for i,tag in enumerate(tags):
    df_y = pd.read_csv(f'{tag}_hiv_benchmark.csv')
    metrics = get_metrics(df_y.y_test, df_y.y_pred, df_y.y_prob)
    print(f'\n{versions[i]}')
    pprint.pprint(metrics)